# Figuras para la Tesis (MIL Multimodal)
Generación de figuras vectorizadas para el documento de tesis, mostrando la alineación geométrica, la atención temporal y los resultados agregados.

In [1]:
import sys
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import traceback

# Configuración LaTeX para Matplotlib
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Computer Modern Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14
})

sys.path.append(str(Path.cwd()))
from equinos import rutas, inferencia, clasificacion, caracteristicas

def main():
    rutas.inicializar()
    dir_figuras = rutas.carpeta("resultados") / "figuras"
    dir_figuras.mkdir(parents=True, exist_ok=True)
    
    # -------------------------------------------------------------------------
    # Figura 1: Preprocesamiento y Alineación
    # -------------------------------------------------------------------------
    try:
        print("\nGenerando Figura 1...")
        video_id = "240_20260903-084041_yW6FmTyX"
        import os
        base_videos = Path(os.environ.get("TESINA_EQUINOS_VIDEOS", rutas.carpeta("videos") / "Videos de prueba"))
        ruta_video = base_videos / "Disgustado" / f"{video_id}.mp4"
        
        if not ruta_video.exists():
            print(f"[Error] No se encontró el video {ruta_video}")
        else:
            recortes, pose_filas = inferencia.preparar_clip(ruta_video)
            from equinos import pose
            puntos = pose.seleccionar_par(pose_filas)
            
            cap = cv2.VideoCapture(str(ruta_video))
            frames_crudos = []
            while True:
                ret, frame = cap.read()
                if not ret: break
                frames_crudos.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            cap.release()
            
            idx_frame = len(recortes) // 2
            frame = frames_crudos[idx_frame]
            
            parametros = inferencia.parametros_etapa02()
            caja_list = inferencia._detectar([cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)], parametros)
            caja = caja_list[0] if caja_list else None
            
            pt = puntos[idx_frame]
            ojo = (pt["x_ojo"], pt["y_ojo"])
            hocico = (pt["x_hocico"], pt["y_hocico"])
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
            
            ax1.imshow(frames_crudos[idx_frame])
            if caja:
                x1, y1, x2, y2 = map(int, caja)
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2)
                ax1.add_patch(rect)
            if ojo and hocico and ojo[0] == ojo[0]: 
                ax1.plot([ojo[0], hocico[0]], [ojo[1], hocico[1]], 'c-', linewidth=2)
                ax1.plot(ojo[0], ojo[1], 'yo', markersize=8, label="Ojo")
                ax1.plot(hocico[0], hocico[1], 'mo', markersize=8, label="Hocico")
                ax1.legend()
            ax1.set_title("Detección (YOLO + DLC)")
            ax1.axis("off")
            
            ax2.imshow(cv2.cvtColor(recortes[idx_frame], cv2.COLOR_BGR2RGB))
            ax2.set_title("Recorte Alineado")
            ax2.axis("off")
            
            plt.tight_layout()
            plt.savefig(dir_figuras / "figura1_preprocesamiento.pdf", bbox_inches='tight')
            plt.savefig(dir_figuras / "figura1_preprocesamiento.png", dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
    except Exception as e:
        print("[Fallo Figura 1]")
        traceback.print_exc()

    # -------------------------------------------------------------------------
    # Figura 2: Secuencia y Pesos de Atención (MIL)
    # -------------------------------------------------------------------------
    try:
        print("\nGenerando Figura 2...")
        if not ruta_video.exists():
            print(f"[Error] No se encontró el video {ruta_video}")
        else:
            resultado = inferencia.clasificar_video(ruta_video, progreso=lambda *args, **kwargs: None)
            atencion = resultado.atencion
            
            if atencion is not None and len(recortes) > 0:
                n_frames = min(16, len(recortes))
                recortes_clip = recortes[:n_frames]
                atencion_clip = atencion[:n_frames]
                
                idx_mostrar = list(range(len(recortes_clip)))
                
                fig = plt.figure(figsize=(24, 6))
                gs = fig.add_gridspec(2, len(idx_mostrar), height_ratios=[1, 1])
                
                for i, idx in enumerate(idx_mostrar):
                    ax = fig.add_subplot(gs[0, i])
                    ax.imshow(cv2.cvtColor(recortes_clip[idx], cv2.COLOR_BGR2RGB))
                    ax.axis('off')
                    ax.set_title(f"t={idx}")
                    
                ax_att = fig.add_subplot(gs[1, :])
                tiempo = np.arange(len(atencion_clip))
                ax_att.bar(tiempo, atencion_clip, color='steelblue', alpha=0.8)
                ax_att.set_xlabel("Fotograma")
                ax_att.set_ylabel("Peso de Atención")
                ax_att.set_xlim(-0.5, len(atencion_clip)-0.5)
                for i, idx in enumerate(idx_mostrar):
                    ax_att.axvline(x=idx, color='red', linestyle='--', alpha=0.5)
                    
                plt.tight_layout()
                plt.savefig(dir_figuras / "figura2_atencion.pdf", bbox_inches='tight')
                plt.savefig(dir_figuras / "figura2_atencion.png", dpi=300, bbox_inches='tight')
                plt.show()
                plt.close()
    except Exception as e:
        print("[Fallo Figura 2]")
        traceback.print_exc()

    # -------------------------------------------------------------------------
    # Figura 3: Espacio Geométrico Normalizado
    # -------------------------------------------------------------------------
    try:
        print("\nGenerando Figura 3...")
        eval_path = rutas.carpeta("resultados") / "objetivo3_evaluacion.json"
        if eval_path.exists():
            from equinos import pose
            pose_filas = pose.leer_pose()
            
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
            ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
            
            import random
            from collections import defaultdict
            pose_por_clip = defaultdict(lambda: defaultdict(list))
            for f in pose_filas:
                pose_por_clip[f['clip']][int(f['orden'])].append(f)
                
            def mejor_de(candidatos, frames_medidos):
                mejor = (float("nan"), float("nan"), -1.0)
                for c in candidatos:
                    for f in frames_medidos:
                        if f['bodypart'].lower() == c.lower() or c.lower() in f['bodypart'].lower():
                            conf = float(f['confianza'])
                            if conf > mejor[2]:
                                mejor = (float(f['x']), float(f['y']), conf)
                return mejor
    
            import math
            puntos_ojo = []
            puntos_hoc = []
            puntos_ore = []
            
            clips_lista = list(pose_por_clip.keys())
            random.seed(42)
            clips_muestra = random.sample(clips_lista, min(100, len(clips_lista)))
            
            for clip in clips_muestra:
                frames_del_clip = pose_por_clip[clip]
                for orden in range(len(frames_del_clip)):
                    medidos = frames_del_clip[orden]
                    x_ojo, y_ojo, c_ojo = mejor_de(["eye"], medidos)
                    x_hoc, y_hoc, c_hoc = mejor_de(["nose", "snout", "muzzle", "upper_jaw", "mouth"], medidos)
                    x_ore, y_ore, c_ore = mejor_de(["earbase", "earend"], medidos)
                    
                    if c_ojo > 0 and c_hoc > 0:
                        cx, cy = (x_ojo + x_hoc) / 2, (y_ojo + y_hoc) / 2
                        escala = math.hypot(x_ojo - x_hoc, y_ojo - y_hoc)
                        if escala < 1e-6: escala = 1.0
                        
                        def normalizar(x, y, c):
                            if c > 0.0 and not math.isnan(x) and not math.isnan(y):
                                return (x - cx) / escala, (y - cy) / escala
                            return None, None
                            
                        nx_ojo, ny_ojo = normalizar(x_ojo, y_ojo, c_ojo)
                        nx_hoc, ny_hoc = normalizar(x_hoc, y_hoc, c_hoc)
                        nx_ore, ny_ore = normalizar(x_ore, y_ore, c_ore)
                        
                        if nx_ojo is not None: puntos_ojo.append((nx_ojo, ny_ojo))
                        if nx_hoc is not None: puntos_hoc.append((nx_hoc, ny_hoc))
                        if nx_ore is not None: puntos_ore.append((nx_ore, ny_ore))
            
            if puntos_ore:
                ox, oy = zip(*puntos_ore)
                ax.scatter(ox, oy, alpha=0.5, color='cyan', label='Orejas', s=10, zorder=2)
            if puntos_ojo:
                ojx, ojy = zip(*puntos_ojo)
                ax.scatter(ojx, ojy, color='yellow', edgecolor='black', s=50, label='Ojos', zorder=5)
            if puntos_hoc:
                hx, hy = zip(*puntos_hoc)
                ax.scatter(hx, hy, color='magenta', edgecolor='black', s=50, label='Hocicos', zorder=5)
            
            ax.invert_yaxis()
            ax.set_xlim(-2.5, 2.5)
            ax.set_ylim(2.5, -2.5) 
            
            ax.set_aspect('equal', adjustable='datalim')
            ax.set_title("Espacio Geométrico Normalizado (Multimodal)")
            ax.set_xlabel("X (normalizado por dist. ojo-hocico)")
            ax.set_ylabel("Y (normalizado por dist. ojo-hocico)")
            ax.legend()
            
            plt.tight_layout()
            plt.savefig(dir_figuras / "figura3_espacio_geometrico.pdf", bbox_inches='tight')
            plt.savefig(dir_figuras / "figura3_espacio_geometrico.png", dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
    except Exception as e:
        print("[Fallo Figura 3]")
        traceback.print_exc()

    # -------------------------------------------------------------------------
    # Figura 4: Matriz de Confusión Final
    # -------------------------------------------------------------------------
    try:
        print("\nGenerando Figura 4...")
        if eval_path.exists():
            datos = json.loads(eval_path.read_text(encoding='utf-8'))
            preds = datos["predicciones_fuera_de_muestra"]
            
            clases = clasificacion.CLASES
            matriz = np.zeros((len(clases), len(clases)), dtype=int)
            mapa_idx = {c: i for i, c in enumerate(clases)}
            
            for p in preds:
                r = mapa_idx[p["real"]]
                pr = mapa_idx[p["predicha"]]
                matriz[r, pr] += 1
                
            fig, ax = plt.subplots(figsize=(6, 5))
            sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', 
                        xticklabels=clases, yticklabels=clases, ax=ax,
                        cbar=False, annot_kws={"size": 12})
            ax.set_xlabel("Predicción del Modelo", fontweight='bold')
            ax.set_ylabel("Clase Real (Anotación)", fontweight='bold')
            ax.set_title("Matriz de Confusión Agregada (4 Folds)", pad=15)
            
            plt.tight_layout()
            plt.subplots_adjust(top=0.88)
            plt.savefig(dir_figuras / "figura4_matriz_confusion.pdf", bbox_inches='tight', pad_inches=0.2)
            plt.savefig(dir_figuras / "figura4_matriz_confusion.png", dpi=300, bbox_inches='tight', pad_inches=0.2)
            plt.show()
            plt.close()
    except Exception as e:
        print("[Fallo Figura 4]")
        traceback.print_exc()

    print("\nFiguras generadas en", dir_figuras)

if __name__ == '__main__':
    main()


Entorno         : local
Raíz de proyecto: C:\Users\hrlic\Desktop\UGR\Lic Ciencia de Datos\Materias\Seminario\Tesina-UGR\datos_equinos
Área de trabajo : C:\Users\hrlic\Desktop\UGR\Lic Ciencia de Datos\Materias\Seminario\Tesina-UGR\datos_equinos\trabajo

Generando Figura 1...


C:\Users\hrlic\Desktop\UGR\Lic Ciencia de Datos\Materias\Seminario\Tesina-UGR\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\hrlic\Desktop\UGR\Lic Ciencia de Datos\Materias\Seminario\Tesina-UGR\env\Lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:139: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)


  0%|          | 0/54 [00:00<?, ?it/s]

  6%|▌         | 3/54 [00:00<00:01, 27.39it/s]

...


Generando Figura 3...


<Figure size 600x600 with 1 Axes>


Generando Figura 4...


<Figure size 600x500 with 1 Axes>


Figuras generadas en C:\Users\hrlic\Desktop\UGR\Lic Ciencia de Datos\Materias\Seminario\Tesina-UGR\datos_equinos\resultados\figuras
